In [0]:
from pathlib import Path
import pandas as pd
import numpy as np

In [0]:
#Leemos la tabla en esquema bronze y transformamos de spark a pandas
df_s = spark.table("workspace.bronze.superstore")
df = df_s.toPandas()

In [0]:
df.columns

In [0]:
print(df['Segment'].value_counts())
print(df['Ship Mode'].value_counts())

In [0]:
#Reemplazamos los espacios de las columnas con _ y renombramos con minusculas"
def clean_column_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace(".", "_").replace("-", "_")
df.columns = [clean_column_name(col) for col in df.columns]
df.columns

In [0]:
df.dtypes

In [0]:
#Validamos que no hayan negativos en campo QUANTITY
df[df['quantity']<0]

In [0]:
#Reemplazamos las comas por puntos, para obtener un número decimal
df['sales']=df['sales'].str.replace(',','.').astype(float)
df['discount']=df['discount'].str.replace(',','.').astype(float)
df['profit']=df['profit'].str.replace(',','.').astype(float)


In [0]:
#Eliminamos row_id
df = df.drop(columns=['row_id'])
#Agregamos la fecha de actualización
df['fec_informacion']=pd.Timestamp.now()
df['fec_informacion'].head(5)

In [0]:
#Guardamos la información en el esquema Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
df_spark = spark.createDataFrame(df)
df_spark.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.superstore")